documents
   ↓
text extraction
   ↓
chunking
   ↓
embeddings
   ↓
AI Search / Vector Search
   ↓
retrieval
   ↓
LLM answer

In [0]:
from pyspark.sql import functions as F

docs = (
    spark.table("workspace.silver.events")
    .select(
        "GlobalEventID",
        "AddedDate",
        "ActionGeo_CountryCode",
        "ActionGeo_FullName",
        "SOURCEURL"
    )
    .filter(F.col("SOURCEURL").isNotNull())
    .filter(F.col("SOURCEURL") != "")
    .dropDuplicates(["SOURCEURL"])
    .limit(1000)
)

print("Candidate documents:", docs.count())
display(docs.limit(10))

In [0]:
import requests
from bs4 import BeautifulSoup
from pyspark.sql import functions as F

# Take a small prototype sample.
url_rows = (
    spark.table("workspace.silver.events")
    .select(
        "GlobalEventID",
        "AddedDate",
        "ActionGeo_CountryCode",
        "SOURCEURL"
    )
    .filter(F.col("SOURCEURL").isNotNull())
    .filter(F.col("SOURCEURL") != "")
    .dropDuplicates(["SOURCEURL"])
    .limit(100)
    .collect()
)

documents = []

for row in url_rows:
    url = row["SOURCEURL"]

    try:
        response = requests.get(
            url,
            timeout=10,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        if response.status_code != 200:
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove things that aren't article content.
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        text = " ".join(soup.stripped_strings)

        # Ignore pages that returned almost no useful text.
        if len(text) < 500:
            continue

        documents.append({
            "doc_id": str(row["GlobalEventID"]),
            "AddedDate": row["AddedDate"],
            "CountryCode": row["ActionGeo_CountryCode"],
            "url": url,
            "text": text[:20000]
        })

    except Exception:
        continue

print("Articles successfully fetched:", len(documents))

In [0]:
docs_df = spark.createDataFrame(documents)

display(
    docs_df.select(
        "doc_id",
        "CountryCode",
        "url",
        "text"
    ).limit(5)
)

In [0]:
(
    docs_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.news_documents")
)

print(
    "Documents:",
    spark.table("workspace.gold.news_documents").count()
)

In [0]:
display(
    spark.sql("""
        SELECT
            doc_id,
            AddedDate,
            CountryCode,
            url,
            LENGTH(text) AS text_length
        FROM workspace.gold.news_documents
        ORDER BY text_length DESC
        LIMIT 20
    """)
)

In [0]:
from pyspark.sql import functions as F

documents = spark.table("workspace.gold.news_documents")

chunks = (
    documents
    .withColumn(
        "chunk_start",
        F.explode(
            F.sequence(
                F.lit(1),
                F.length("text"),
                F.lit(800)
            )
        )
    )
    .withColumn(
        "chunk_text",
        F.substring(
            "text",
            F.col("chunk_start"),
            1200
        )
    )
    .filter(F.length("chunk_text") >= 200)
    .withColumn(
        "chunk_id",
        F.concat_ws(
            "_",
            F.col("doc_id"),
            F.col("chunk_start")
        )
    )
    .select(
        "chunk_id",
        "doc_id",
        "AddedDate",
        "CountryCode",
        "url",
        "chunk_text"
    )
)

print("Chunks:", chunks.count())

display(chunks.limit(10))

In [0]:
(
    chunks.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.news_chunks")
)

print(
    "Saved chunks:",
    spark.table("workspace.gold.news_chunks").count()
)

In [0]:
%pip install databricks-ai-search

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.ai_search.client import AISearchClient

client = AISearchClient()
print("AI Search SDK loaded")

In [0]:
endpoint = client.create_endpoint(
    name="gdelt-rag-search"
)

print(endpoint)

In [0]:
index = client.create_delta_sync_index(
    endpoint_name="gdelt-rag-search",
    index_name="workspace.gold.news_chunks_index",
    source_table_name="workspace.gold.news_chunks",
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="chunk_text",
    embedding_model_endpoint_name="databricks-gte-large-en"
)

print(index)

In [0]:
%sql
ALTER TABLE workspace.gold.news_chunks SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
index = client.get_index(
    endpoint_name="gdelt-rag-search",
    index_name="workspace.gold.news_chunks_index"
)

print(index.describe())


In [0]:
import time

while True:
    index = client.get_index(
        endpoint_name="gdelt-rag-search",
        index_name="workspace.gold.news_chunks_index"
    )
    status = index.describe()["status"]
    print(f"Index status: {status['detailed_state']} — {status['message']}")
    if status["ready"]:
        break
    time.sleep(10)

index.sync()
print("Sync triggered")

In [0]:
endpoint = client.get_endpoint("gdelt-rag-search")

print(endpoint)

In [0]:
index = client.get_index(
    endpoint_name="gdelt-rag-search",
    index_name="workspace.gold.news_chunks_index"
)

print(index.describe())

In [0]:
index.sync()
print("Sync triggered")

In [0]:
results = index.similarity_search(
    query_text="recent news about the United States",
    columns=[
        "chunk_id",
        "doc_id",
        "CountryCode",
        "url",
        "chunk_text"
    ],
    num_results=5
)

display(results)

In [0]:
results = index.similarity_search(
    query_text="news about political events and government activity in the United States",
    columns=[
        "chunk_id",
        "CountryCode",
        "url",
        "chunk_text"
    ],
    num_results=5
)

display(results)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# ------------------------------------------------------------
# 1. Retrieve relevant chunks
# ------------------------------------------------------------

query = "What recent news activity is being reported in the United States?"

results = index.similarity_search(
    query_text=query,
    columns=[
        "chunk_id",
        "CountryCode",
        "url",
        "chunk_text"
    ],
    num_results=5
)

# ------------------------------------------------------------
# 2. Build context from retrieved chunks
# ------------------------------------------------------------

columns = [col["name"] for col in results["manifest"]["columns"]]
rows = [dict(zip(columns, vals)) for vals in results["result"]["data_array"]]

context_parts = []

for i, row in enumerate(rows, start=1):
    context_parts.append(
        f"""
SOURCE {i}
Country: {row.get("CountryCode", "")}
URL: {row.get("url", "")}

TEXT:
{row.get("chunk_text", "")}
"""
    )

context = "\n\n".join(context_parts)

# ------------------------------------------------------------
# 3. Ask a Databricks-hosted LLM to answer using the context
# ------------------------------------------------------------

w = WorkspaceClient()

response = w.serving_endpoints.query(
    name="databricks-llama-4-maverick",
    messages=[
        ChatMessage(
            role=ChatMessageRole.SYSTEM,
            content=(
                "You are a news intelligence assistant. "
                "Answer the user's question using ONLY the supplied "
                "retrieved context. If the context is insufficient, "
                "say so. Do not invent facts. "
                "Mention the relevant source URLs when useful."
            )
        ),
        ChatMessage(
            role=ChatMessageRole.USER,
            content=f"""
Question:
{query}

Retrieved context:
{context}
"""
        )
    ],
    max_tokens=500
)

answer = response.choices[0].message.content

print(answer)

**We are here**

                         GDELT
                           │
              ┌────────────┴────────────┐
              │                         │
       Event Database              News documents
              │                         │
              ▼                         ▼
          BRONZE                    Documents
              │                         │
              ▼                         ▼
          SILVER                    Chunks
              │                         │
              ▼                         ▼
           GOLD                 AI Search index
              │                         │
       ┌──────┴──────┐                  ▼
       │             │               Retrieval
       ▼             ▼                  │
 Analytics       ML Features            ▼
                       │               LLM
                       ▼                │
                 Isolation Forest       ▼
                       │              RAG answer
                       ▼
                    MLflow
                       │
                       ▼
                Model Registry
                       │
                       ▼
                 Model Serving
                       │
                       ▼
                    REST API

I built a GDELT-based lakehouse on Databricks. Raw event data was ingested into a governed Bronze layer, transformed and validated in Silver, and aggregated into Gold analytics tables. I engineered country-level temporal features such as rolling event volume and deviation metrics, trained an unsupervised Isolation Forest to detect unusual event-volume behavior, tracked the experiment with MLflow, registered the model in Unity Catalog, and deployed it through Model Serving. Separately, I built a RAG prototype by extracting article text, chunking documents, generating embeddings through Databricks AI Search, retrieving relevant chunks, and passing the retrieved context to an LLM.